<a href="https://colab.research.google.com/github/lawrennd/qig-code/blob/main/examples/qutrit_gibbs_lock_clock_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qutrit Gibbs-Lock Experiments: $\Pi_{\text{marg}}$, Loewner Kernel, and Hamiltonian Clock

This notebook implements **CIP-000F**: five experiments for the $d=3$ qutrit system that concretely verify the Hamiltonian clock construction derived in  
**"Gibbs-Lock and the Emergence of Hamiltonian Structure in the Inaccessible Game"** (Lawrence-hamiltonian26).

## Physical system

The running example throughout the paper is a single qutrit described by the Hamiltonian

$$H_\delta = \mathrm{diag}(0,\,0,\,\delta), \qquad K_0 = \beta_0 H_\delta,$$

with Gibbs state

$$\rho_0 = \frac{1}{Z}\,\mathrm{diag}(1,\,1,\,e^{-\beta_0\delta}), \qquad Z = 2 + e^{-\beta_0\delta}.$$

The spectrum has **one degenerate block** $\{\lambda_0 = \lambda_1 = 1/Z\}$ and **one separated level** $\{\lambda_2 = e^{-\beta_0\delta}/Z\}$, yielding two qualitatively different off-diagonal mode types:

| Mode type | Indices | Bohr gap | Analytical Loewner weight |
|-----------|---------|----------|---------------------------|
| In-block (degen.) | $(0,1)$, $(1,0)$ | $0$ | $1/Z$ |
| Cross-block | $(0,2)$, $(1,2)$, c.c. | $\delta$ | $(1-e^{-\beta_0\delta})/(Z\beta_0\delta)$ |

**Note on bipartite structure.** Experiments 1 and 2 require the marginal entropy constraint, which needs a bipartite partition. Those experiments use `near_bell_gibbs_frame(d=3)` (a $3\times 3 = 9$-dimensional joint system). Experiments 3, 4 and 5 operate on the single-qutrit 3×3 system described above, which is where the paper's closed forms hold exactly.

## Experiments

| # | Title | Key claim |
|---|-------|-----------|
| 3 | Loewner kernel two-sector structure | $C_{01}=1/Z$; $C_{02}=(1-e^{-\beta_0\delta})/(Z\beta_0\delta)$; smooth $\delta\to 0$ limit |
| 5 | $\hbar(\beta_0,\delta)$ closed form | $\hbar = Z^2/(2\beta_0^2\delta^2 e^{-\beta_0\delta})$; minimum near $x=\beta_0\delta\approx 2.66$ |
| 2 | Iso-marginal sector check | $\Pi_{\text{marg}}(K_0)\,R_{\text{od}} = R_{\text{od}}$ for all off-diagonal generators |
| 4 | Uniform dephasing | Decay rate $\mu_0 = c\hbar$ uniform across both mode types |
| 1 | Explicit $\Pi_{\text{marg}}$ | Rank-78 projector; complement spanned by constraint gradients |

---
## Setup

In [ ]:
# Auto-install QIG package if not available
try:
    import qig
except ImportError:
    print('Installing QIG package...')
    %pip install -q git+https://github.com/lawrennd/qig-code.git
    import qig

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

from qig.gibbs_lock import GibbsLockedFrame, infer_mu0
from qig.pair_operators import near_bell_gibbs_frame

plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'figure.dpi': 120,
})
print('qig version:', qig.__version__ if hasattr(qig, '__version__') else 'dev')

### Physical setup: the departed qutrit

We construct the single-qutrit `GibbsLockedFrame` with $H_\delta = \mathrm{diag}(0,0,\delta)$ at reference parameters $\delta=0.5$, $\beta_0=2.0$.

In [ ]:
# Reference parameters
DELTA_REF = 0.5
BETA_REF  = 2.0

def departed_qutrit_frame(delta: float, beta: float) -> GibbsLockedFrame:
    """Single-qutrit GibbsLockedFrame with H = diag(0, 0, delta)."""
    H = np.diag([0.0, 0.0, delta])
    return GibbsLockedFrame(H, beta=beta)

def Z_qutrit(delta: float, beta: float) -> float:
    """Partition function Z = 2 + exp(-beta*delta)."""
    return 2.0 + np.exp(-beta * delta)

frame_ref = departed_qutrit_frame(DELTA_REF, BETA_REF)
Z_ref = Z_qutrit(DELTA_REF, BETA_REF)

rho0 = frame_ref.rho0
vals_rho, _ = np.linalg.eigh(rho0)
vals_rho = vals_rho[::-1]  # descending

print(f'Reference: delta={DELTA_REF}, beta={BETA_REF}')
print(f'Z = {Z_ref:.6f}')
print()
print('rho_0 eigenvalues (descending):')
for i, v in enumerate(vals_rho):
    print(f'  lambda_{i} = {v:.6f}')
print()
print('Expected  lambda_0 = lambda_1 = 1/Z =', round(1/Z_ref, 6))
print('Expected  lambda_2 = exp(-beta*delta)/Z =', round(np.exp(-BETA_REF*DELTA_REF)/Z_ref, 6))
print()
print('Gibbs-lock residual ||[K0, H]||_F =', frame_ref.gibbs_lock_residual())

---
## Experiment 3 — Loewner kernel two-sector structure

**Goal.** Verify the analytical Loewner weights for both off-diagonal mode types and confirm the smooth $\delta\to 0$ degenerate limit.

The `loewner_kernel()` method returns the matrix $C$ in the eigenbasis of $\rho_0$, where $C_{ij} = k(\lambda_i, \lambda_j)$ with the BKM divided-difference kernel

$$k(p,q) = \frac{p - q}{\log p - \log q} \quad (p \ne q), \qquad k(p,p) = p.$$

The eigenvalues returned by `loewner_kernel()` are **sorted ascending**, so for our system:

| Sorted index | Paper index | Value |
|:---:|:---:|---|
| 0 | 2 | $\lambda_2 = e^{-\beta_0\delta}/Z$ (separated level) |
| 1 | 0 | $\lambda_0 = 1/Z$ (degenerate pair) |
| 2 | 1 | $\lambda_1 = 1/Z$ (degenerate pair) |

Correspondingly:
- **In-block** entry: $C[1,2] = 1/Z$
- **Cross-block** entry: $C[0,1] = C[0,2] = (1-e^{-\beta_0\delta})/(Z\beta_0\delta)$

In [ ]:
# --- Verify at the reference point ---
C, vals_sorted, vecs = frame_ref.loewner_kernel()

w_inblock_numerical  = C[1, 2].real  # in-block (degenerate pair, sorted indices 1,2)
w_cross_numerical    = C[0, 1].real  # cross-block (separated vs degenerate, sorted index 0 vs 1)

w_inblock_analytical  = 1.0 / Z_ref
w_cross_analytical    = (1 - np.exp(-BETA_REF * DELTA_REF)) / (Z_ref * BETA_REF * DELTA_REF)

print('--- Loewner weights at reference point ---')
print(f'In-block  C[1,2]:  numerical = {w_inblock_numerical:.10f}')
print(f'           analytic = {w_inblock_analytical:.10f}')
print(f'           error    = {abs(w_inblock_numerical - w_inblock_analytical):.2e}')
print()
print(f'Cross-block C[0,1]: numerical = {w_cross_numerical:.10f}')
print(f'            analytic = {w_cross_analytical:.10f}')
print(f'            error    = {abs(w_cross_numerical - w_cross_analytical):.2e}')

# Assertions
assert abs(w_inblock_numerical - w_inblock_analytical) < 1e-10, "In-block weight mismatch"
assert abs(w_cross_numerical   - w_cross_analytical)   < 1e-10, "Cross-block weight mismatch"
print()
print('✓ Both weights match analytical closed forms to 1e-10')

In [ ]:
# --- Verify over a (beta, delta) grid ---
betas  = np.linspace(0.5, 5.0, 10)
deltas = np.logspace(-2, np.log10(2.0), 20)  # log-spaced to probe delta -> 0 limit

max_err_inblock = 0.0
max_err_cross   = 0.0

for beta in betas:
    for delta in deltas:
        f = departed_qutrit_frame(delta, beta)
        C_grid, _, _ = f.loewner_kernel()
        Z = Z_qutrit(delta, beta)

        # In-block: sorted indices 1,2 are the degenerate pair
        w_in_num  = C_grid[1, 2].real
        w_in_ana  = 1.0 / Z
        max_err_inblock = max(max_err_inblock, abs(w_in_num - w_in_ana))

        # Cross-block: sorted index 0 (separated) vs sorted index 1 (degenerate)
        w_cr_num  = C_grid[0, 1].real
        w_cr_ana  = (1 - np.exp(-beta * delta)) / (Z * beta * delta)
        max_err_cross = max(max_err_cross, abs(w_cr_num - w_cr_ana))

print(f'Grid size: {len(betas)} x {len(deltas)} = {len(betas)*len(deltas)} points')
print(f'Max error in-block:   {max_err_inblock:.2e}')
print(f'Max error cross-block: {max_err_cross:.2e}')

assert max_err_inblock < 1e-10, f"In-block weight error too large: {max_err_inblock:.2e}"
assert max_err_cross   < 1e-10, f"Cross-block weight error too large: {max_err_cross:.2e}"
print()
print('✓ Both weights verified over full (beta, delta) grid to 1e-10')

In [ ]:
# --- Verify smooth delta -> 0 limit ---
# lim_{delta->0} (1 - exp(-beta*delta)) / (Z * beta * delta)  =  1/Z|_{delta=0} = 1/3
beta_check = 2.0
tiny_deltas = np.logspace(-6, -1, 30)
cross_weights  = []
inblock_weights = []

for delta in tiny_deltas:
    f = departed_qutrit_frame(delta, beta_check)
    Cg, _, _ = f.loewner_kernel()
    cross_weights.append(Cg[0, 1].real)
    inblock_weights.append(Cg[1, 2].real)

# At delta=0: Z=3, so both limits should equal 1/3
limit_expected = 1.0 / 3.0
err_limit = abs(cross_weights[0] - limit_expected)
print(f'Cross-block weight at delta={tiny_deltas[0]:.1e}: {cross_weights[0]:.10f}')
print(f'In-block  weight at delta={tiny_deltas[0]:.1e}: {inblock_weights[0]:.10f}')
print(f'Expected limit 1/3 = {limit_expected:.10f}')
print(f'Cross-block error from limit: {err_limit:.2e}')

assert err_limit < 1e-4, f"delta->0 limit not approached: error = {err_limit:.2e}"
print()
print('✓ Cross-block weight converges to in-block weight (1/Z|_{delta=0} = 1/3) as delta -> 0')

In [ ]:
# --- Plot: Loewner weights vs x = beta * delta (fixing beta) ---
beta_plot = 2.0
x_vals    = np.logspace(-2, 1, 200)  # x = beta * delta
delta_plot = x_vals / beta_plot

w_inblock_plot = []
w_cross_plot   = []
w_analytic_inblock = []
w_analytic_cross   = []

for delta in delta_plot:
    f = departed_qutrit_frame(delta, beta_plot)
    Cg, _, _ = f.loewner_kernel()
    w_inblock_plot.append(Cg[1, 2].real)
    w_cross_plot.append(Cg[0, 1].real)
    Z = Z_qutrit(delta, beta_plot)
    w_analytic_inblock.append(1.0 / Z)
    w_analytic_cross.append((1 - np.exp(-beta_plot * delta)) / (Z * beta_plot * delta))

fig, ax = plt.subplots(figsize=(6, 4))

ax.semilogx(x_vals, w_inblock_plot,       'C0-',  lw=2, label=r'In-block $C_{12}$ (numerical)')
ax.semilogx(x_vals, w_analytic_inblock,   'C0--', lw=1.5, alpha=0.6, label=r'In-block $1/Z$ (analytic)')
ax.semilogx(x_vals, w_cross_plot,         'C1-',  lw=2, label=r'Cross-block $C_{01}$ (numerical)')
ax.semilogx(x_vals, w_analytic_cross,     'C1--', lw=1.5, alpha=0.6,
            label=r'Cross-block $(1-e^{-x})/(Zx)$ (analytic)')

ax.axhline(1/3, color='gray', ls=':', lw=1)
ax.text(0.012, 1/3 + 0.004, r'$1/3\;(\delta\to 0$ limit)', color='gray', fontsize=9)

ax.set_xlabel(r'$x = \beta_0\delta$')
ax.set_ylabel('Loewner weight')
ax.set_title(r'Exp 3: Loewner kernel two-sector structure ($\beta_0=2$)')
ax.legend(fontsize=9)
ax.set_xlim(x_vals[0], x_vals[-1])
ax.set_ylim(0, 0.45)
plt.tight_layout()
plt.savefig('fig_exp3_loewner-kernel-two-sector.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: fig_exp3_loewner-kernel-two-sector.pdf')

### Exp 3 — Summary

- **In-block weight** $C_{12} = 1/Z$: constant for all $\delta$, confirmed to $10^{-10}$.
- **Cross-block weight** $C_{01} = (1-e^{-x})/(Zx)$ with $x = \beta_0\delta$: decreases monotonically in $x$, confirmed to $10^{-10}$.
- **Smooth degenerate limit**: as $\delta\to 0$, the cross-block weight converges to the in-block weight $1/Z|_{\delta=0} = 1/3$.

Gap-dependent amplitude suppression enters through the Loewner kernel when $\delta>0$, while the decay rate remains uniform in modular-generator coordinates (verified in Exp 4).